# MENG 417 — Homework #6
## Complete worked solutions and Fashion-MNIST convolutional neural network

**Source:** Professor's *MENG 417 Homework #6*, two-page handout. This notebook preserves the original question order:

- **Question 1:** one-dimensional convolution, then operation without flipping the filter and comparison.
- **Question 2:** two-dimensional convolution of the specified image and filter, calculated by hand.
- **Question 3:** Fashion-MNIST classifier with **at least three convolutional layers**, training/validation loss curves for **up to 10 epochs**, a confusion matrix, and a comparison against fully connected networks, AlexNet, and GoogLeNet.

**Final deliverable:** The professor asks for **one PDF**. After running all cells, use the PDF-export cell at the end to generate a report containing the calculations, *actual measured* CNN results, plots, confusion matrix, architecture explanation, and Python source appendix.

**Mathematics:** All displayed formulas use paired dollar-sign math delimiters throughout.

**Academic accuracy:** Questions 1–2 have fixed, verifiable numeric answers. Question 3's test accuracy, loss values, and confusion matrix require actually training the CNN; these outputs are computed, not prewritten or guessed.

# Part I — Hand-calculation questions

## Convolution versus correlation: the essential distinction

A discrete convolution **reverses the filter relative to the sliding operation**:

$$
\boxed{y[n]=(x*h)[n]=\sum_{k=-\infty}^{\infty}h[k]x[n-k].}
$$

The professor uses this equivalent form. Outside the supplied signal ranges, assume zero-valued samples.

If we slide the filter **without reversing its coefficients**, the operation is *cross-correlation*. The full discrete cross-correlation under the convention used here is

$$
\boxed{r_{xh}[\ell]=\sum_{k=-\infty}^{\infty}x[\ell+k]h[k].}
$$

A reversed filter only leads to the same answer if its coefficients are symmetric in the relevant sense. The professor's filter $$[1,2]$$ is not symmetric, so the two outputs differ.

**Analogy:** Convolution slides a mirror-reflected stencil; correlation slides the stencil as drawn.

In [ ]:
# Common imports for the hand calculations and plots.
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import convolve, correlate, convolve2d, correlate2d

np.set_printoptions(linewidth=130, suppress=True)

# Question 1 — One-dimensional convolution (show your work)

The professor supplies the signals, beginning at index 0:

$$
\boxed{x=[1,2,3,4],\quad x[0]=1}
$$

$$
\boxed{h=[1,2],\quad h[0]=1.}
$$

The convolution is

$$
y[n]=\sum_{k=0}^{1}h[k]x[n-k]
     =x[n]+2x[n-1],
$$

with $$x[n]=0$$ outside $$0\le n\le3$$.

The **full** convolution has

$$
N_y=N_x+N_h-1=4+2-1=5
$$

samples, at indices $$n=0,1,2,3,4$$.

## Calculate every output sample

**At $$n=0$$:**

$$y[0]=1x[0]+2x[-1]=1+0=\boxed{1}.$$

**At $$n=1$$:**

$$y[1]=1x[1]+2x[0]=2+2=\boxed{4}.$$

**At $$n=2$$:**

$$y[2]=1x[2]+2x[1]=3+4=\boxed{7}.$$

**At $$n=3$$:**

$$y[3]=1x[3]+2x[2]=4+6=\boxed{10}.$$

**At $$n=4$$:**

$$y[4]=1x[4]+2x[3]=0+8=\boxed{8}.$$

### Final convolution answer

$$
\boxed{(x*h)[n]=[1,\ 4,\ 7,\ 10,\ 8],\qquad n=0,1,2,3,4.}
$$

In [ ]:
# Question 1: independent manual calculation and SciPy verification.
x_1d = np.array([1, 2, 3, 4], dtype=np.int64)
h_1d = np.array([1, 2], dtype=np.int64)

manual_conv = []
for n in range(len(x_1d) + len(h_1d) - 1):
    val = 0
    for k in range(len(h_1d)):
        j = n - k
        if 0 <= j < len(x_1d):
            val += int(h_1d[k] * x_1d[j])
    manual_conv.append(val)

conv_1d = convolve(x_1d, h_1d, mode="full", method="direct")
assert manual_conv == [1, 4, 7, 10, 8]
np.testing.assert_array_equal(conv_1d, manual_conv)
print("Hand calculation:", manual_conv)
print("SciPy full convolution:", conv_1d.tolist())

# Question 1(a) — Without flipping the filter

When the filter is applied without flipping, we calculate **cross-correlation**. Its sliding products follow the *unreversed* tap order $$h[0]=1$$, $$h[1]=2$$:

$$
\boxed{r_{xh}[\ell]=x[\ell]\cdot 1+x[\ell+1]\cdot 2.}
$$

Because the signals are finite, full correlation has **five** samples. Under the convention above, their lags are

$$\ell=-1,0,1,2,3.$$

Compute each lag:

$$
\begin{aligned}
r_{xh}[-1]&=0(1)+1(2)=\boxed{2},\\
r_{xh}[0]&=1(1)+2(2)=\boxed{5},\\
r_{xh}[1]&=2(1)+3(2)=\boxed{8},\\
r_{xh}[2]&=3(1)+4(2)=\boxed{11},\\
r_{xh}[3]&=4(1)+0(2)=\boxed{4}.
\end{aligned}
$$

### Final no-flip answer

$$
\boxed{r_{xh}[\ell]=[2,\ 5,\ 8,\ 11,\ 4]\text{ for }\ell=-1,0,1,2,3.}
$$

**Compare with convolution:**

| Operation | Full output | Associated indices |
|---|---|---|
| Convolution (filter reversed during sliding) | $$[1,4,7,10,8]$$ | $$n=0,1,2,3,4$$ |
| Cross-correlation (no flip) | $$[2,5,8,11,4]$$ | $$\ell=-1,0,1,2,3$$ |

The outputs differ because the filter $$[1,2]$$ is not symmetric. Cross-correlation is often used under the name "convolution" in machine-learning software, although mathematically the operation does not flip the kernel.

**Boundary note:** If one considers only fully overlapping windows without padding (*valid* mode), the unflipped outputs would instead be $$[5,8,11]$$. Here we consistently use **full mode** to compare like with like.

In [ ]:
# Question 1(a): correlation verification, including correct lag indices.
corr_1d = correlate(x_1d, h_1d, mode="full", method="direct")
correlation_lags = np.arange(-(len(h_1d)-1), len(x_1d))
np.testing.assert_array_equal(corr_1d, [2, 5, 8, 11, 4])
np.testing.assert_array_equal(correlation_lags, [-1, 0, 1, 2, 3])
valid_no_flip = correlate(x_1d, h_1d, mode="valid", method="direct")
print("Convolution (full):", conv_1d.tolist())
print("Correlation (full):", corr_1d.tolist())
print("Correlation lags:  ", correlation_lags.tolist())
print("Correlation (valid):", valid_no_flip.tolist())

In [ ]:
# Visual comparison: the same finite signals, two different operations.
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), constrained_layout=True)
axes[0].stem(range(5), conv_1d)
axes[0].set_title("True convolution (full)")
axes[0].set_xlabel("Output index n")
axes[0].set_ylabel("Value")
axes[0].grid(alpha=0.25)
axes[1].stem(correlation_lags, corr_1d)
axes[1].set_title("Cross-correlation (no filter flip)")
axes[1].set_xlabel("Lag l")
axes[1].grid(alpha=0.25)
plt.show()

# Question 2 — Two-dimensional convolution (show your work)

The professor gives the image

$$
X=\begin{bmatrix}
-4&1&-3\\
3&-2&-3\\
-4&2&-3
\end{bmatrix}
$$

and the filter

$$
H=\begin{bmatrix}
1&0&0\\
0&0&1\\
0&0&0
\end{bmatrix}.
$$

The handout says **"you are free to select the axes"** but does not specify a boundary convention. We choose:

- zero-based row/column indices starting at the **top-left**;
- row index increases downward, column index to the right;
- zero-valued data outside the 3-by-3 image;
- **full 2D convolution**, so the output is $$5\times5$$.

Under these declared conventions, every answer is reproducible.

## Step 1 — Write the 2D convolution formula

$$
\boxed{Y[i,j]=\sum_{a=0}^2\sum_{b=0}^2H[a,b]X[i-a,j-b].}
$$

## Step 2 — Identify the only two nonzero taps

The only nonzero coefficients are

$$H[0,0]=1,\qquad H[1,2]=1.$$

Therefore the double sum simplifies **exactly** to

$$
\boxed{Y[i,j]=X[i,j]+X[i-1,j-2],}
$$

where an out-of-bounds entry of $$X$$ is interpreted as zero.

This is the shortest reliable way to compute all 25 output cells by hand.

## Question 2 — Detailed row-by-row calculation

The full output indices are $$i,j\in\{0,1,2,3,4\}$$.

**Row $$i=0$$:** only $$X[0,j]$$ contributes:

$$Y[0,:]=[-4,\ 1,\ -3,\ 0,\ 0].$$

**Row $$i=1$$:** $$X[1,j]$$ and the $$X[0,j-2]$$ shifted contribution:

$$
\begin{aligned}
Y[1,0]&=3+0=3,\\
Y[1,1]&=-2+0=-2,\\
Y[1,2]&=-3+(-4)=-7,\\
Y[1,3]&=0+1=1,\\
Y[1,4]&=0+(-3)=-3.
\end{aligned}
$$

**Row $$i=2$$:**

$$
\begin{aligned}
Y[2,0]&=-4,\quad Y[2,1]=2,\\
Y[2,2]&=-3+3=0,\\
Y[2,3]&=0+(-2)=-2,\\
Y[2,4]&=0+(-3)=-3.
\end{aligned}
$$

**Row $$i=3$$:** the shifted $$X[2,j-2]$$ contributes:

$$Y[3,:]=[0,\ 0,\ -4,\ 2,\ -3].$$

**Row $$i=4$$:** both contributions are zero:

$$Y[4,:]=[0,\ 0,\ 0,\ 0,\ 0].$$

### Final 2D convolution answer (full mode)

$$
\boxed{
Y_{\mathrm{full}}=\begin{bmatrix}
-4&1&-3&0&0\\
3&-2&-7&1&-3\\
-4&2&0&-2&-3\\
0&0&-4&2&-3\\
0&0&0&0&0
\end{bmatrix}.}
$$

The chosen boundary convention matters: in the common "same" mode the corresponding centered output is the central 3-by-3 portion of this full result, and "valid" mode gives one value.

In [ ]:
# Question 2 — calculate by hand (using the two non-zero filter taps).
X = np.array([[-4, 1, -3], [3, -2, -3], [-4, 2, -3]], dtype=np.int64)
H = np.array([[1, 0, 0], [0, 0, 1], [0, 0, 0]], dtype=np.int64)

Y_manual = np.zeros((5, 5), dtype=np.int64)
for i in range(5):
    for j in range(5):
        direct = int(X[i, j]) if i < 3 and j < 3 else 0
        shifted = int(X[i - 1, j - 2]) if 0 <= i - 1 < 3 and 0 <= j - 2 < 3 else 0
        Y_manual[i, j] = direct + shifted

Y_full = convolve2d(X, H, mode="full")
Y_same = convolve2d(X, H, mode="same")
Y_valid = convolve2d(X, H, mode="valid")

expected = np.array([
    [-4,  1, -3,  0,  0],
    [ 3, -2, -7,  1, -3],
    [-4,  2,  0, -2, -3],
    [ 0,  0, -4,  2, -3],
    [ 0,  0,  0,  0,  0]
])
np.testing.assert_array_equal(Y_manual, expected)
np.testing.assert_array_equal(Y_full, expected)
np.testing.assert_array_equal(Y_same, expected[1:4, 1:4])
np.testing.assert_array_equal(Y_valid, [[0]])

print("Question 2 — Full convolution (5 by 5):")
print(Y_full)
print("\nSame convolution (3 by 3):")
print(Y_same)
print("\nValid convolution (1 by 1):")
print(Y_valid)

## Question 2 — Verify the role of filter flipping

The operation above is the **true convolution** asked for in the question. If we were to slide $$H$$ without flipping it, we would obtain *cross-correlation*, which is a different 5-by-5 matrix.

This matters in neural networks: a `Conv2D` layer in TensorFlow/Keras uses the deep-learning **cross-correlation** convention (no spatial filter flip), although the API calls the layer a convolution.

For hand calculations, follow the mathematical convolution definition unless the instructor says **not to flip**.

In [ ]:
# Supplementary visual: mathematical convolution vs no-flip correlation.
Y_correlation = correlate2d(X, H, mode="full")
print("True 2D convolution:\n", Y_full)
print("Unflipped 2D cross-correlation:\n", Y_correlation)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.3), constrained_layout=True)
for ax, arr, title in zip(
    axes,
    [Y_full, Y_correlation],
    ["True full convolution", "Full cross-correlation (no flip)"]
):
    im = ax.imshow(arr, cmap="coolwarm", vmin=-7, vmax=7)
    ax.set_title(title)
    ax.set_xlabel("Column j")
    ax.set_ylabel("Row i")
    ax.set_xticks(range(5))
    ax.set_yticks(range(5))
    for (i, j), v in np.ndenumerate(arr):
        ax.text(j, i, str(v), ha="center", va="center", fontsize=9)
plt.show()

# Part II — Question 3: Fashion-MNIST CNN

The professor requires:

1. Build a Fashion-MNIST recognizer with **at least three convolutional layers**.
2. Train it and plot **training and validation loss curves for up to 10 epochs**.
3. Explain the difference between this CNN and a **fully connected network**, **AlexNet**, and **GoogLeNet**.
4. Provide the **Python code and confusion matrix**.

We will use **exactly three `Conv2D` layers**, with `MaxPooling2D` for spatial reduction, followed by a small dense classifier. Our model is purpose-built for grayscale **28-by-28** Fashion-MNIST images; it is **not** a reproduction of AlexNet or GoogLeNet.

## How Fashion-MNIST is organized

The official dataset has:

$$
60{,}000\text{ training images},\quad 10{,}000\text{ independent test images}.
$$

Each image has $$28\times28$$ grayscale pixels belonging to one of 10 clothing classes. We will reserve **12,000** of the original training images for validation, leaving **48,000** for gradient updates. The independent test set is not used for training or hyperparameter selection.

The TensorFlow-supported loader for this dataset is:

```python
(x_train_all, y_train_all), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
```

The Homework #6 handout does **not** prohibit framework dataset loaders. The raw `.gz` loading restriction from the earlier Homework #4 is not part of this handout.

In [ ]:
# Q3 — Reproducible TensorFlow setup. Run this in Google Colab.
# Google Colab usually has TensorFlow installed already.
import os
import time
import tensorflow as tf

SEED = 417
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
print("TensorFlow version:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))
print("Note: CPU execution is valid; selecting a Colab GPU may improve speed.")

In [ ]:
# Q3 — Load Fashion-MNIST, create a held-out validation split, and normalize.
(x_train_all_u8, y_train_all), (x_test_u8, y_test) = tf.keras.datasets.fashion_mnist.load_data()
assert x_train_all_u8.shape == (60000, 28, 28)
assert x_test_u8.shape == (10000, 28, 28)

# Fixed random split: 48,000 train, 12,000 validation.
indices = np.random.default_rng(SEED).permutation(len(x_train_all_u8))
train_indices, val_indices = indices[:48000], indices[48000:]

x_train = (x_train_all_u8[train_indices].astype("float32") / 255.0)[..., np.newaxis]
y_train = y_train_all[train_indices]
x_val = (x_train_all_u8[val_indices].astype("float32") / 255.0)[..., np.newaxis]
y_val = y_train_all[val_indices]
x_test = (x_test_u8.astype("float32") / 255.0)[..., np.newaxis]

assert x_train.shape == (48000, 28, 28, 1)
assert x_val.shape == (12000, 28, 28, 1)
assert x_test.shape == (10000, 28, 28, 1)
assert not np.intersect1d(train_indices, val_indices).size

CLASS_NAMES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

print("Train:", x_train.shape, "Validation:", x_val.shape, "Test:", x_test.shape)
print("Training pixel range:", float(x_train.min()), "to", float(x_train.max()))

In [ ]:
# Visual reference: one real Fashion-MNIST sample from each class.
fig, axes = plt.subplots(2, 5, figsize=(11, 5), constrained_layout=True)
for cls, ax in enumerate(axes.flat):
    ix = int(np.flatnonzero(y_train == cls)[0])
    ax.imshow(x_train[ix, :, :, 0], cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"{cls}: {CLASS_NAMES[cls]}", fontsize=9)
    ax.axis("off")
plt.show()

## Question 3 — CNN structure and the mathematics of each layer

The CNN uses this structure:

$$
\boxed{28\!\times\!28\!\times\!1
\to\mathrm{Conv}(32,3\times3)
\to\mathrm{Pool}(2\times2)
\to\mathrm{Conv}(64,3\times3)
\to\mathrm{Pool}(2\times2)
\to\mathrm{Conv}(128,3\times3)
\to\mathrm{Pool}(2\times2)
\to\mathrm{Flatten}
\to\mathrm{Dense}(64)
\to\mathrm{Dense}(10).}
$$

**All three convolutions use stride 1 and `padding='same'`.** Hidden activations use ReLU; the final Dense layer uses Softmax for ten-class prediction.

### Spatial shapes

| Stage | Spatial output | Explanation |
|---|---|---|
| Image | $$28\times28\times1$$ | One grayscale channel |
| Conv 1 | $$28\times28\times32$$ | Same padding, 32 filters |
| Max pool 1 | $$14\times14\times32$$ | 2-by-2 downsampling |
| Conv 2 | $$14\times14\times64$$ | 64 learned filters |
| Max pool 2 | $$7\times7\times64$$ | Spatial dimensions halved |
| Conv 3 | $$7\times7\times128$$ | 128 learned filters |
| Max pool 3 | $$3\times3\times128$$ | Floor downsampling of 7 by 2 |
| Flatten | $$1152$$ | $$3\times3\times128$$ |
| Dense hidden | $$64$$ | Learn class-related combinations |
| Softmax | $$10$$ | One class probability per clothing category |

### Stride and padding

For a convolution with input height $$H$$, kernel size $$K$$, padding $$P$$ and stride $$S$$, the standard output-size formula is

$$
\boxed{H_{\mathrm{out}}=\left\lfloor\frac{H+2P-K}{S}\right\rfloor+1.}
$$

For `same` convolution with $$S=1$$, the output height stays the same. A 2-by-2 max-pool with stride 2 and `valid` padding reduces $$28\to14\to7\to3$$ as shown.

### Convolution weight sharing

A 3-by-3 filter taking $$C_{\mathrm{in}}$$ input channels and producing $$C_{\mathrm{out}}$$ output channels has

$$
\boxed{(3\cdot3\cdot C_{\mathrm{in}}+1)C_{\mathrm{out}}}
$$

learnable parameters including biases. The same filters are reused across spatial locations, enabling *local pattern detection* with fewer weights than fully connecting every output pixel to every input pixel.

In [ ]:
# Q3 — Exactly THREE convolutional layers, with pooling.
# Model choice: a compact CNN for 28x28 grayscale images.
def build_fashion_cnn():
    model = tf.keras.Sequential([
        tf.keras.Input(shape=(28, 28, 1)),
        tf.keras.layers.Conv2D(32, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal"),
        tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
        tf.keras.layers.Conv2D(64, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal"),
        tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
        tf.keras.layers.Conv2D(128, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal"),
        tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation="relu", kernel_initializer="he_normal"),
        tf.keras.layers.Dense(10, activation="softmax"),
    ], name="FashionMNIST_3Conv_CNN")
    return model

model = build_fashion_cnn()
model.summary()
conv_layers = [layer for layer in model.layers if isinstance(layer, tf.keras.layers.Conv2D)]
assert len(conv_layers) == 3, "The professor requires at least three convolutional layers."
print("Exactly three convolutional layers:", len(conv_layers))
print("Trainable parameters:", model.count_params())
assert model.count_params() == 167114

## Question 3 — Verify the parameter count

These values follow the architecture above, including each bias term.

**Conv 1:**

$$P_1=(3\cdot3\cdot1+1)\cdot32=\boxed{320}.$$

**Conv 2:**

$$P_2=(3\cdot3\cdot32+1)\cdot64=\boxed{18{,}496}.$$

**Conv 3:**

$$P_3=(3\cdot3\cdot64+1)\cdot128=\boxed{73{,}856}.$$

**Dense hidden:**

$$P_4=(1152+1)\cdot64=\boxed{73{,}792}.$$

**Dense output:**

$$P_5=(64+1)\cdot10=\boxed{650}.$$

**MaxPooling2D**, **Flatten**, and activation functions add no trainable parameters.

$$
\boxed{P_{\mathrm{total}}=320+18{,}496+73{,}856+73{,}792+650=167{,}114.}
$$

## Question 3 — Learning algorithm, loss, optimizer and data processing

Our design is explicitly reproducible:

| Item | Specification |
|---|---|
| Dataset | Fashion-MNIST, 10 classes |
| Training/validation/test | 48,000 / 12,000 / 10,000 |
| Pixel processing | `float32` and pixel value divided by 255 |
| Conv2D layers | 3 (32, 64, 128 filters) |
| Kernel, stride, padding | 3-by-3, stride 1, `same` |
| Pooling | 2-by-2 max-pooling after each convolution |
| Loss | Sparse categorical cross-entropy |
| Optimizer | Adam, initial learning rate $$\eta=0.001$$ |
| Batch size | 128 |
| Epochs | **10**, matching the professor's limit |
| Hidden activation | ReLU |
| Output activation | Softmax |
| Random seed | 417, for improved reproducibility |

The ten Softmax outputs are probabilities $$p_0,\ldots,p_9$$, with

$$\sum_{c=0}^9 p_c=1.$$

For each training observation with integer class label $$y_i$$ and predicted probability $$p_{i,y_i}$$, the sparse cross-entropy loss is

$$
\boxed{\mathcal L=-\frac{1}{N}\sum_{i=1}^{N}\ln p_{i,y_i}.}
$$

Using `Dense(10, activation='softmax')` means the loss must be configured with **`from_logits=False`** (the Keras default). Do not set `from_logits=True` here.

The training set controls the model's weights. The validation set tracks generalization during the 10 epochs. The untouched test set measures final performance after training.

In [ ]:
# Q3 — Compile and train for precisely 10 epochs.
MAX_EPOCHS = 10
BATCH_SIZE = 128
LEARNING_RATE = 1e-3

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=["accuracy"],
)

start_time = time.perf_counter()
history = model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    batch_size=BATCH_SIZE,
    epochs=MAX_EPOCHS,
    shuffle=True,
    verbose=2,
)
training_seconds = time.perf_counter() - start_time
print(f"Training duration: {training_seconds:.2f} seconds")
print("Epochs completed:", len(history.history["loss"]))
assert len(history.history["loss"]) == 10

## Question 3 — Training and validation loss curves

The professor requires both loss curves for up to 10 epochs. Interpret them as follows:

- **Training loss decreasing:** the model is learning patterns in the training images.
- **Validation loss decreasing:** the patterns also generalize to unseen validation images.
- **Training loss decreasing while validation loss rises:** possible overfitting.
- **Both losses staying high:** possible underfitting, insufficient training, or an optimization issue.

The **actual trend must be read from the plotted results**, rather than assumed.

In [ ]:
# Q3 — Show and SAVE the required loss curves for the PDF submission.
epoch_numbers = np.arange(1, len(history.history["loss"]) + 1)
fig, ax = plt.subplots(figsize=(8.4, 4.6))
ax.plot(epoch_numbers, history.history["loss"], "-o", label="Training loss")
ax.plot(epoch_numbers, history.history["val_loss"], "-o", label="Validation loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Sparse categorical cross-entropy")
ax.set_title("Fashion-MNIST CNN — Training and validation loss")
ax.set_xticks(epoch_numbers)
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
LOSS_FIGURE = "HW6_Loss_Curves.png"
fig.savefig(LOSS_FIGURE, dpi=180, bbox_inches="tight")
plt.show()
print("Saved:", LOSS_FIGURE)

In [ ]:
# Optional complementary figure: training and validation accuracy.
fig, ax = plt.subplots(figsize=(8.4, 4.6))
ax.plot(epoch_numbers, history.history["accuracy"], "-o", label="Training accuracy")
ax.plot(epoch_numbers, history.history["val_accuracy"], "-o", label="Validation accuracy")
ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy (fraction)")
ax.set_xticks(epoch_numbers)
ax.set_title("Fashion-MNIST CNN — Accuracy across 10 epochs")
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
ACCURACY_FIGURE = "HW6_Accuracy_Curves.png"
fig.savefig(ACCURACY_FIGURE, dpi=180, bbox_inches="tight")
plt.show()

## Question 3 — Independent test evaluation and confusion matrix

The model is now fully trained. Evaluate the **10,000 test images** only after training is complete.

Define:

$$
\boxed{\mathrm{TestAccuracy}=\frac{\#\text{correct predictions}}{10{,}000}.}
$$

The confusion matrix has shape $$10\times10$$, with

$$
\boxed{C_{i,j}=\#\{\text{true label }i,\ \text{predicted label }j\}.}
$$

Rows correspond to **true classes** and columns to **predicted classes**. The diagonal entries represent correct predictions. Every Fashion-MNIST test class contains 1,000 examples; the confusion-matrix row sums will reflect this.

In [ ]:
# Q3 — True held-out test metrics and confusion matrix; no fabricated values.
test_loss, test_accuracy = model.evaluate(x_test, y_test, batch_size=256, verbose=0)
probabilities = model.predict(x_test, batch_size=256, verbose=0)
y_pred = np.argmax(probabilities, axis=1)
cm = tf.math.confusion_matrix(y_test, y_pred, num_classes=10, dtype=tf.int32).numpy()
correct_count = int(np.trace(cm))
cm_accuracy = correct_count / int(cm.sum())

assert cm.shape == (10, 10)
assert int(cm.sum()) == 10000
assert np.all(cm.sum(axis=1) == 1000)
np.testing.assert_allclose(cm_accuracy, test_accuracy, rtol=0, atol=5e-6)

print(f"Test loss:       {test_loss:.6f}")
print(f"Test accuracy:   {100*test_accuracy:.2f}%")
print(f"Correct predictions: {correct_count}/10000")
print("Confusion matrix (rows=true classes, columns=predicted classes):")
print(cm)

In [ ]:
# Q3 — Labelled confusion-matrix plot, saved for the professor's PDF.
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm, cmap="Blues")
ax.set_title("Fashion-MNIST — Test confusion matrix")
ax.set_xlabel("Predicted class")
ax.set_ylabel("True class")
ax.set_xticks(range(10), labels=CLASS_NAMES, rotation=45, ha="right")
ax.set_yticks(range(10), labels=CLASS_NAMES)
for (i, j), v in np.ndenumerate(cm):
    color = "white" if v > cm.max() * 0.52 else "black"
    ax.text(j, i, str(int(v)), ha="center", va="center", color=color, fontsize=7)
fig.colorbar(im, ax=ax, shrink=0.78, label="Number of test images")
fig.tight_layout()
CM_FIGURE = "HW6_Confusion_Matrix.png"
fig.savefig(CM_FIGURE, dpi=180, bbox_inches="tight")
plt.show()
print("Saved:", CM_FIGURE)

In [ ]:
# Additional quality check: per-class accuracy (recall) from the confusion matrix.
class_support = cm.sum(axis=1)
class_correct = np.diag(cm)
class_accuracy = class_correct / class_support
print("Class-wise accuracy / recall on the test set:")
for name, count, score in zip(CLASS_NAMES, class_support, class_accuracy):
    print(f"{name:12s}  n={int(count):4d}  accuracy={100*score:5.2f}%")
print(f"Macro-average recall: {100*np.mean(class_accuracy):.2f}%")

# Question 3 — Comparison against three other architectures

## (i) Our CNN versus a fully connected neural network

**Fully connected network:** the input image is flattened into 784 numbers, and each Dense neuron connects to every input coordinate. The network does not explicitly preserve the original 2D pixel-neighborhood layout or reuse a small spatial filter at different image locations.

**Our CNN:** the input retains its $$28\times28\times1$$ geometry. Three local convolutional filters progressively learn visual features such as edges, textures and clothing parts, using **weight sharing**. Pooling reduces spatial dimensions, and the final dense layers classify the extracted features.

Example fully connected alternative (for illustration only):

$$784\to128\to64\to10.$$

Its parameter count would be

$$
(784+1)128+(128+1)64+(64+1)10=\boxed{109{,}386}.
$$

Our CNN contains **167,114** total trainable parameters, because it also has a dense classification head. **Do not claim that a CNN must always have fewer total parameters than any fully connected network.** Its key architectural efficiency is *localized connectivity and shared filter weights*, not an unconditional smaller total count.

## (ii) Our CNN versus AlexNet

**AlexNet (2012)** is a historically important large-scale CNN for color ImageNet images; its original architecture has **five convolutional layers** followed by **three fully connected layers** (counting the final classification layer), with pooling, ReLU, dropout in the fully connected portion, and roughly **60 million trainable parameters**. It was developed for significantly larger, higher-resolution RGB images and 1,000 output classes.

Our Fashion-MNIST CNN is much smaller: three convolutional layers, small 3-by-3 filters, grayscale 28-by-28 images and 10 classes. It is not AlexNet, and its accuracy cannot be directly compared with AlexNet's original ImageNet benchmark.

Original AlexNet paper: https://proceedings.neurips.cc/paper_files/paper/2012/file/c399862d3b9d6b76c8436e924a68c45b-Paper.pdf

## (iii) Our CNN versus GoogLeNet (Inception v1)

**GoogLeNet / Inception (2014)** is an approximately **22-weight-layer** CNN that uses **Inception modules** to process the same features through multiple branches (including 1-by-1, 3-by-3 and 5-by-5 convolutions, plus pooling). Outputs are concatenated, and **1-by-1 convolutions** can reduce channel dimensions before computationally expensive branches. GoogLeNet also uses global average pooling and auxiliary classifiers in the training architecture.

Our CNN uses **one linear chain of three convolutional layers**, each followed by pooling. It has no parallel Inception modules, no multi-branch feature concatenation and no auxiliary classifiers.

Original GoogLeNet paper: https://research.google/pubs/going-deeper-with-convolutions/

### Quick comparison table

| Model | Relevant distinction | Intended context |
|---|---|---|
| Fully connected | Global Dense connections; no explicitly shared local image filters | Generic vector/tabular or flattened input |
| **Our CNN** | 3 sequential Conv2D layers + max pooling + Dense classifier | 28×28 Fashion-MNIST, 10 classes |
| AlexNet | 5 conv + 3 fully connected, large historical architecture | Large RGB ImageNet classification |
| GoogLeNet | Deep Inception modules with parallel multi-scale filters | Efficient deep ImageNet image recognition |

The correct conclusion is that the CNN is more **spatially structured** than a fully connected classifier, considerably **smaller and shallower** than classic AlexNet, and much **simpler** than multi-branch GoogLeNet.

## Question 3 — Interpretation of your real training curves

The professor asks how the model improved during training. Because the measured trends depend on the actual run, the next cell builds an interpretation directly from the learning-history data and reports the final results.

Useful observations to include in a final report:

- How training cross-entropy changed from epoch 1 to epoch 10.
- How validation cross-entropy changed over the same period.
- Whether the minimum validation loss occurred before epoch 10 (potential overfitting if validation loss later rises).
- The independently measured test accuracy and loss.
- Which test labels are most often confused (read from the off-diagonal confusion-matrix entries).

These are observations, not guarantees of constant improvement every epoch.

In [ ]:
# Q3 — Data-driven conclusions; use these outputs in the final report.
train_loss_values = np.asarray(history.history["loss"], dtype=float)
val_loss_values = np.asarray(history.history["val_loss"], dtype=float)
best_val_epoch = int(np.argmin(val_loss_values) + 1)
train_loss_change = float(train_loss_values[-1] - train_loss_values[0])
val_loss_change = float(val_loss_values[-1] - val_loss_values[0])

confusions = cm.copy()
np.fill_diagonal(confusions, 0)
worst_i, worst_j = np.unravel_index(np.argmax(confusions), confusions.shape)

print("Question 3: Final experimental conclusions")
print(f"Training loss:   {train_loss_values[0]:.4f} -> {train_loss_values[-1]:.4f} (change {train_loss_change:+.4f})")
print(f"Validation loss: {val_loss_values[0]:.4f} -> {val_loss_values[-1]:.4f} (change {val_loss_change:+.4f})")
print(f"Best validation-loss epoch: {best_val_epoch} / {MAX_EPOCHS}")
print(f"Test accuracy: {test_accuracy*100:.2f}%  |  Test loss: {test_loss:.5f}")
print(f"Most common incorrect pair: true {CLASS_NAMES[worst_i]} -> predicted {CLASS_NAMES[worst_j]} ({int(confusions[worst_i, worst_j])} cases)")
if val_loss_values[-1] > val_loss_values[best_val_epoch-1] + 1e-5:
    print("Validation loss increased after its best epoch: consider this possible overfitting evidence, not proof.")
else:
    print("No clear late increase over the best observed validation loss.")

# Final answer summary — Questions 1, 2, 3

## Q1

Mathematical full convolution:

$$\boxed{[1,\ 4,\ 7,\ 10,\ 8]}.$$

Without flipping the filter (full cross-correlation):

$$\boxed{[2,\ 5,\ 8,\ 11,\ 4]}.$$

The two results differ because $$h=[1,2]$$ is not symmetric.

## Q2

Under **full**, zero-padded, top-left-indexed 2D convolution:

$$
\boxed{\begin{bmatrix}
-4&1&-3&0&0\\
3&-2&-7&1&-3\\
-4&2&0&-2&-3\\
0&0&-4&2&-3\\
0&0&0&0&0
\end{bmatrix}}.
$$

Under **same** mode, the central 3-by-3 output is:

$$
\begin{bmatrix}
-2&-7&1\\
2&0&-2\\
0&-4&2
\end{bmatrix}.
$$

## Q3

- **Required architecture:** implemented and verified 3 Conv2D layers.
- **Ten epochs:** training and validation loss curves saved in the notebook.
- **Confusion matrix:** generated and labelled from the actual 10,000 independent test predictions.
- **Comparisons:** fully connected, AlexNet, and GoogLeNet explained above.
- **Accuracy:** reported dynamically from the trained model. Do not write a test accuracy into your submission until this notebook has executed on real Fashion-MNIST data.

# Theory cheat sheet — MENG 417 Homework #6

| Concept | Formula or rule | Meaning |
|---|---|---|
| 1D convolution | $$y[n]=\sum_k h[k]x[n-k]$$ | Filter reversed during sliding |
| 1D cross-correlation | $$r[\ell]=\sum_k h[k]x[\ell+k]$$ | Same tap order as printed |
| Full 1D length | $$N_x+N_h-1$$ | Includes zero-padded edge overlaps |
| 2D convolution | $$Y[i,j]=\sum_{a,b}H[a,b]X[i-a,j-b]$$ | Spatial filtering with flipped kernel |
| Full 2D shape | $$(H_x+H_h-1)\times(W_x+W_h-1)$$ | Here 5-by-5 |
| Kernel | Small 2D weight matrix | Detects local spatial patterns |
| Stride | Movement of filter between positions | Large stride reduces spatial size |
| Padding | Border convention; e.g. `same` or `valid` | Controls output size/boundary behavior |
| ReLU | $$\max(0,z)$$ | Nonlinear hidden activation |
| MaxPooling2D | Max in each spatial pooling window | Reduces image feature-map size |
| Weight sharing | Same kernel at every spatial location | Helps detect features independent of location |
| Sparse cross-entropy | $$-\frac1N\sum_i\log p_{i,y_i}$$ | Loss for integer-labeled classes |
| Softmax | $$p_c=e^{z_c}/\sum_j e^{z_j}$$ | Normalized class probabilities |
| Confusion matrix | $$C_{ij}=\#(y=i,\hat y=j)$$ | True classes in rows; predictions in columns |
| Test accuracy | $$\mathrm{trace}(C)/\sum C$$ | Correct classifications / all test samples |
| AlexNet | 5 conv + 3 FC | Early large ImageNet CNN |
| GoogLeNet | Inception multi-branch modules | Deeper, multi-scale CNN |

## Frequent grading mistakes

1. Treating mathematical convolution and cross-correlation as identical even with a non-symmetric filter.
2. Writing a 3-by-3 answer for Q2 without explaining why full/same/valid mode was chosen.
3. Using a CNN with only two Conv2D layers when the professor requires **at least three**.
4. Plotting training **accuracy** when the professor specifically asks for training and validation **loss curves**.
5. Swapping the true and predicted axes of the confusion matrix.
6. Reporting plausible but **unmeasured** Fashion-MNIST accuracy.
7. Comparing our Fashion-MNIST test score numerically with AlexNet or GoogLeNet's ImageNet result (different datasets).
8. Forgetting the professor's **single PDF with code** submission requirement.

# Verification checkpoint before submission

The checks below validate fixed arithmetic, model design, actual training history, confusion-matrix arithmetic, and the professor's requested outputs. If any assertion fails, investigate it before generating the final PDF.

In [ ]:
# Full verification of the current notebook's real outputs.
np.testing.assert_array_equal(conv_1d, [1, 4, 7, 10, 8])
np.testing.assert_array_equal(corr_1d, [2, 5, 8, 11, 4])
np.testing.assert_array_equal(Y_full, expected)
np.testing.assert_array_equal(Y_same, expected[1:4, 1:4])
assert len([l for l in model.layers if isinstance(l, tf.keras.layers.Conv2D)]) >= 3
assert len(history.history["loss"]) == 10
assert len(history.history["val_loss"]) == 10
assert cm.shape == (10, 10) and int(cm.sum()) == 10000
assert abs(float(np.trace(cm)) / 10000 - float(test_accuracy)) < 5e-6
for f in [LOSS_FIGURE, ACCURACY_FIGURE, CM_FIGURE]:
    assert os.path.exists(f), f"Missing required plot: {f}"
print("All checks passed. The PDF exporter may now be run.")

# Export one professor-submission PDF, including the code appendix

The professor's handout explicitly requires **one PDF**. After executing every preceding cell, run the following cell. It creates and downloads:

**`MENG417_Homework6_Final_Submission.pdf`**

The report includes complete Q1 and Q2 calculations, the Q3 neural-network specification, **measured** test accuracy/loss, training and validation loss plot, labeled confusion matrix, architecture comparison, and the executed code cells as a source appendix.

If you edit the notebook, use **Runtime → Run all** again before exporting, so the report matches the current code and actual training outputs. This cell will refuse to proceed if the required experiment outputs do not exist.

In [ ]:
# Build the professor-required ONE PDF using the actual outputs from this run.
# ReportLab is used only for exporting the report; it does not affect the CNN.
import subprocess
import sys
import textwrap
from pathlib import Path

try:
    from reportlab.lib.pagesizes import A4
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "reportlab"])

from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import mm
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
    Image, PageBreak, Preformatted
)
from xml.sax.saxutils import escape

for varname in ["model", "history", "cm", "test_accuracy", "test_loss", "Y_full"]:
    if varname not in globals():
        raise RuntimeError(f"Missing experimental result {varname}. Run the notebook top to bottom first.")
for path in [LOSS_FIGURE, CM_FIGURE, ACCURACY_FIGURE]:
    if not Path(path).exists():
        raise FileNotFoundError(f"Missing figure {path}; rerun the plotting cells before PDF export.")

PDF_NAME = "MENG417_Homework6_Final_Submission.pdf"
styles = getSampleStyleSheet()
styles.add(ParagraphStyle(name="HW6Title",parent=styles["Title"],fontSize=19,leading=23,spaceAfter=12))
styles.add(ParagraphStyle(name="HW6Heading",parent=styles["Heading2"],fontSize=12,leading=15,spaceBefore=11,spaceAfter=5))
styles.add(ParagraphStyle(name="HW6Body",parent=styles["BodyText"],fontSize=9.5,leading=13,spaceAfter=6))
styles.add(ParagraphStyle(name="HW6Code",fontName="Courier",fontSize=6.6,leading=8))

story = []
def para(s): story.append(Paragraph(s, styles["HW6Body"]))
def heading(s): story.append(Paragraph(s, styles["HW6Heading"]))
def table_grid(rows, widths=None):
    t=Table([[str(v) for v in r] for r in rows],colWidths=widths, repeatRows=1, hAlign="CENTER")
    t.setStyle(TableStyle([
        ("BACKGROUND", (0,0),(-1,0),colors.HexColor("#E8EDF4")),
        ("GRID",(0,0),(-1,-1),0.4,colors.HexColor("#9BA8B9")),
        ("VALIGN",(0,0),(-1,-1),"MIDDLE"),
        ("FONTSIZE",(0,0),(-1,-1),8),
        ("TOPPADDING",(0,0),(-1,-1),5),
        ("BOTTOMPADDING",(0,0),(-1,-1),5),
    ]))
    story.append(t)
    story.append(Spacer(1,8))

def matrix_table(a):
    a=np.asarray(a)
    table_grid([["row/col"]+[str(j) for j in range(a.shape[1])]]+
        [[str(i)]+[str(int(v)) for v in row] for i,row in enumerate(a)])

story.append(Paragraph("MENG 417 — Homework #6",styles["HW6Title"]))
para("Complete mathematical solutions and Fashion-MNIST CNN experimental report")

heading("Question 1 — 1D Convolution")
para("Given x = [1,2,3,4] and h = [1,2] beginning at index zero. Zero outside the input range.")
para("True convolution: y[n] = x[n] + 2x[n-1], for n = 0,1,2,3,4.")
table_grid([
    ["n","calculation","y[n]"],
    [0,"1 + 0",1], [1,"2 + 2",4], [2,"3 + 4",7],
    [3,"4 + 6",10],[4,"0 + 8",8]
])
para("<b>Final convolution:</b> [1, 4, 7, 10, 8].")
heading("Question 1(a) — No-flip Cross-Correlation")
para("Use r[l] = x[l] + 2x[l+1]. Full correlation lags are l = -1,0,1,2,3.")
table_grid([
    ["lag","calculation","r[lag]"],
    [-1,"0 + 2",2],[0,"1 + 4",5],[1,"2 + 6",8],
    [2,"3 + 8",11],[3,"4 + 0",4]
])
para("<b>Final cross-correlation:</b> [2, 5, 8, 11, 4]. It differs from convolution because [1,2] is not symmetric.")

heading("Question 2 — 2D Convolution")
para("Choose zero-based axes from the top-left, zero values outside the image, and full convolution.")
para("The only nonzero filter entries are H[0,0] = 1 and H[1,2] = 1; therefore Y[i,j] = X[i,j] + X[i-1,j-2].")
para("Input image X:")
matrix_table(X)
para("Filter H:")
matrix_table(H)
para("<b>Final full 5-by-5 output:</b>")
matrix_table(Y_full)
para("The corresponding centered SAME output is:")
matrix_table(Y_same)

heading("Question 3 — Fashion-MNIST CNN")
para("Dataset: 48,000 training images, 12,000 validation images, and 10,000 untouched test images. Images normalized to [0,1].")
para("Architecture: 28x28x1 -> Conv2D(32,3x3) -> MaxPool -> Conv2D(64,3x3) -> MaxPool -> Conv2D(128,3x3) -> MaxPool -> Flatten -> Dense(64) -> Dense(10,Softmax).")
para(f"The model has exactly {len(conv_layers)} convolutional layers and {model.count_params():,} trainable parameters.")
para(f"Training: Adam, learning rate {LEARNING_RATE}, sparse categorical cross-entropy, batch size {BATCH_SIZE}, {MAX_EPOCHS} epochs.")
para(f"<b>MEASURED held-out test accuracy: {100*test_accuracy:.2f}% ({correct_count}/10,000 correct).</b>")
para(f"<b>MEASURED test loss: {test_loss:.6f}.</b> Best validation-loss epoch: {best_val_epoch}.")
para(f"Training loss: {train_loss_values[0]:.4f} to {train_loss_values[-1]:.4f}; validation loss: {val_loss_values[0]:.4f} to {val_loss_values[-1]:.4f}.")

heading("Training and validation loss (10 epochs)")
story.append(Image(LOSS_FIGURE, width=166*mm, height=91*mm,kind="proportional"))
heading("Independent test confusion matrix")
story.append(Image(CM_FIGURE, width=164*mm, height=129*mm,kind="proportional"))
para("Rows indicate the true class and columns the predicted class; the diagonal contains correctly classified examples.")

heading("Architectural comparisons")
para("<b>Fully connected:</b> Dense layers flatten the pixel grid and use global connections. Our CNN preserves spatial neighborhoods and shares small filters over different positions, with pooling for spatial downsampling.")
para("<b>AlexNet (2012):</b> a large historical ImageNet CNN with five convolutional and three fully connected layers, designed for larger RGB images; our Fashion-MNIST CNN uses three convolutional layers and ten output classes.")
para("<b>GoogLeNet (Inception v1):</b> an approximately 22-weight-layer network with parallel multi-scale Inception branches and dimensionality-reducing 1x1 convolutions. Our CNN is a simple sequential stack without Inception branches.")

story.append(PageBreak())
heading("Appendix — Complete executed Python code")
para("The following code was collected from the executed notebook cells before this PDF-export cell. Run the notebook top to bottom so the printed appendix matches the experiment.")

ip = get_ipython() if "get_ipython" in globals() else None
if ip is None:
    raise RuntimeError("Code-history export requires a Jupyter or Google Colab notebook session.")
raw_history = ip.history_manager.input_hist_raw
source_cells = []
for h in raw_history:
    h=h.strip()
    if not h or h.startswith("%") or h.startswith("!"):
        continue
    if "PDF_NAME = \"MENG417_Homework6_Final_Submission.pdf\"" in h:
        continue
    if h.startswith("#") and not any(t in h for t in ["import ", "def ", "assert", "="]):
        continue
    source_cells.append(h)
if len(source_cells)<12:
    raise RuntimeError("Too few code cells in execution history. Use Runtime -> Run all before export.")

for cell_number, source in enumerate(source_cells, 1):
    story.append(Paragraph(f"Code cell {cell_number}",styles["HW6Body"]))
    lines=[]
    for line in source.splitlines():
        if len(line)>110:
            lines.extend(textwrap.wrap(line,width=105,break_long_words=False,drop_whitespace=False))
        else:
            lines.append(line)
    for k in range(0,len(lines),54):
        story.append(Preformatted("\n".join(lines[k:k+54]),styles["HW6Code"],maxLineLength=110))
        story.append(Spacer(1,4))

doc=SimpleDocTemplate(PDF_NAME,pagesize=A4,leftMargin=16*mm,rightMargin=16*mm,topMargin=14*mm,bottomMargin=14*mm)
def page_footer(canvas,document):
    canvas.saveState()
    canvas.setFont("Helvetica",8)
    canvas.setFillColor(colors.grey)
    canvas.drawRightString(A4[0]-16*mm,8*mm,f"Page {document.page}")
    canvas.restoreState()
doc.build(story,onFirstPage=page_footer,onLaterPages=page_footer)
print("Created:",PDF_NAME,"|",Path(PDF_NAME).stat().st_size,"bytes")
try:
    from google.colab import files
    files.download(PDF_NAME)
except ImportError:
    print("The PDF is available in the current Jupyter working directory.")